# 05 — Direct Preference Optimization (DPO)

**Módulo:** `src/llm_rlhf/dpo.py` → `DPOTrainer`

DPO (Rafailov et al., 2023) colapsa todo el pipeline de RLHF en una única pérdida supervisada. Sin modelo de recompensa, sin rollouts, sin estimación de ventajas. Los mismos datos de preferencias que PPO; el mismo checkpoint de partida; un solo bucle en lugar de tres.

## La derivación, en cuatro líneas

El objetivo de RLHF regularizado por KL es:

$$\max_\pi \; \mathbb{E}_{x \sim \mathcal{D},\, y \sim \pi}\big[ r(x, y) \big] - \beta \, \text{KL}\big( \pi(\cdot | x) \,\|\, \pi_{\text{ref}}(\cdot | x) \big)$$

Tiene una solución óptima cerrada:

$$\pi^*(y | x) = \frac{1}{Z(x)} \pi_{\text{ref}}(y | x) \exp\!\left( \frac{r(x, y)}{\beta} \right)$$

Despejando $r$:

$$r(x, y) = \beta \log \frac{\pi^*(y|x)}{\pi_{\text{ref}}(y|x)} + \beta \log Z(x)$$

El término por prompt $\beta \log Z(x)$ es el mismo para toda respuesta al mismo prompt, así que **se cancela** en cualquier comparación por pares.

## Sustituir en Bradley–Terry

Recuerda del notebook 03: $P(y_w \succ y_l) = \sigma(r(y_w) - r(y_l))$. Sustituye el $r$ reparametrizado:

$$P(y_w \succ y_l \,|\, x) = \sigma\!\left( \beta \log \frac{\pi^*(y_w|x)}{\pi_{\text{ref}}(y_w|x)} - \beta \log \frac{\pi^*(y_l|x)}{\pi_{\text{ref}}(y_l|x)} \right)$$

Minimizar la log-verosimilitud negativa da la pérdida DPO:

$$L_{\text{DPO}}(\theta) = -\mathbb{E}\!\left[ \log \sigma\!\Big( \beta \big( \log\frac{\pi_\theta(y_w|x)}{\pi_{\text{ref}}(y_w|x)} - \log\frac{\pi_\theta(y_l|x)}{\pi_{\text{ref}}(y_l|x)} \big) \Big) \right]$$

Esto es simplemente Bradley–Terry con la *recompensa implícita* $\beta \log(\pi_\theta / \pi_{\text{ref}})$ en lugar de un modelo de recompensa explícito.

## Qué desaparece

Comparado con RLHF + PPO:

- Modelo de recompensa (sin entrenamiento separado)
- Decisiones arquitectónicas del modelo de recompensa
- Reward hacking (la recompensa es implícita en el policy)
- Paso de rollout (sin muestreo on-policy)
- Función de valor y GAE
- Lógica de clipping
- Regularizador de entropía

Lo que queda: un policy de referencia congelado (todavía necesario) y los pares de preferencias (todavía necesarios).

## La pérdida en código

Esta es la actualización DPO completa; todo lo demás en `dpo.py` es fontanería de datos y bookkeeping.

In [ ]:
import torch, torch.nn.functional as F

def dpo_loss(pol_lp_w, pol_lp_l, ref_lp_w, ref_lp_l, beta=0.1):
    # implicit reward = beta * (log pi_theta - log pi_ref)
    w_ratio = pol_lp_w - ref_lp_w   # preferred
    l_ratio = pol_lp_l - ref_lp_l   # dispreferred
    logits = beta * (w_ratio - l_ratio)
    return -F.logsigmoid(logits).mean()

# Preferred is 1 logit more likely under the policy than under reference;
# dispreferred is 1 logit less likely. Loss should be small.
print(dpo_loss(torch.tensor(1.0), torch.tensor(-1.0),
               torch.tensor(0.0), torch.tensor(0.0), beta=0.1).item())

## El rol de β

`β` controla cuán agresivamente confiamos en la señal de preferencia:

- **β grande** — brechas grandes de recompensa implícita, updates grandes, el policy puede alejarse mucho de la referencia (bueno si las preferencias son confiables).
- **β pequeño** — recompensas implícitas pequeñas, updates conservadores, el policy se queda cerca del SFT (bueno si las preferencias son ruidosas).

El valor por defecto `β = 0.1` en `configs/dpo.toml` es un punto de partida común. En la literatura de RLHF, β corresponde al coeficiente de la penalización KL en PPO.

### β, visualizado

La siguiente celda grafica la pérdida DPO en función del *margen* de log-ratio (`(log π/π_ref)[y_w] - (log π/π_ref)[y_l]`) para varios valores de β. Lo que verás:

- Todas las curvas se cruzan en margen = 0 con pérdida $\log 2 \approx 0.693$ (modelo indeciso).
- β más grande hace la pérdida *más empinada*: el mismo margen empuja más fuerte.
- β cero degenera: la pérdida ya no distingue entre $y_w$ e $y_l$.

In [ ]:
import matplotlib.pyplot as plt
import numpy as np

# Plot DPO loss as a function of the log-ratio margin, for several betas.
# margin = (log pi_theta/pi_ref)[y_w] - (log pi_theta/pi_ref)[y_l]
margin = np.linspace(-5, 5, 300)

fig, ax = plt.subplots(figsize=(7.5, 4.5))
for beta in [0.05, 0.1, 0.3, 1.0]:
    losses = [-np.log(1 / (1 + np.exp(-beta * m))) for m in margin]
    ax.plot(margin, losses, label=f'β = {beta}', linewidth=2)
ax.axvline(0, color='gray', alpha=0.4)
ax.set_xlabel('log-ratio margin   (log π/π_ref)[y_w] − (log π/π_ref)[y_l]')
ax.set_ylabel('DPO loss')
ax.set_title('DPO loss vs. implicit-reward margin, for several β')
ax.legend()
ax.grid(alpha=0.3)
plt.tight_layout()
plt.show()

## DPO usa los mismos pares de preferencias que el modelo de recompensa

La entrada a DPO es *exactamente* los datos que usarías para entrenar un modelo de recompensa — tripletas `(prompt, y_w, y_l)`. Así que si ya entrenaste un modelo de recompensa (notebook 03), puedes entrenar DPO sobre los mismos pares y comparar:

```python
pairs = reward_trainer.create_preference_dataset(sft.generate, prompts)
dpo = DPOTrainer(sft)
dpo.train(pairs)
```

In [ ]:
from llm_rlhf.dpo import DPOTrainer, DPOConfig
from llm_rlhf.config import load_toml

cfg = DPOConfig(**load_toml('../configs/dpo.toml'))
print(cfg)

## ¿Cuándo DPO rinde peor que PPO?

La pérdida DPO se calcula solo sobre el dataset *offline* de preferencias. PPO muestrea respuestas *nuevas* en cada iteración y obtiene señales de recompensa frescas. Por tanto:

- DPO brilla cuando las preferencias son abundantes y de alta calidad.
- PPO brilla cuando la exploración on-policy importa — p. ej. tareas donde el modelo necesita descubrir comportamientos que el dataset no muestra.

En la práctica, DPO se ha vuelto el estándar para instruction tuning; PPO sigue siendo común para RLHF-de-AI-feedback y tareas de razonamiento.

## Ejercicio: barrido de β sobre un batch fijo

Un detalle sutil de DPO: para un batch dado de preferencias, β solo escala la *magnitud* de la pérdida, no cambia *qué* preferencias se satisfacen (eso depende solo del signo del margen).

La siguiente celda hace un mini-batch sintético con tres ejemplos de margen conocido, y barre β. Antes de correrla, predice:

- ¿Cuándo crece más rápido la pérdida con β: cuando los márgenes son pequeños o cuando son grandes?
- Si todos los márgenes ya son positivos (modelo correcto), ¿β grande ayuda o estorba?

In [ ]:
# Exercise: how does the DPO loss respond to a β sweep on a fixed batch?
# Make a small batch with a *known* preference margin, vary β, and watch the loss.
import torch

batch = {
    'pol_w': torch.tensor([+0.5, +1.2, -0.1]),   # log π_θ on preferred  (in log-ratio units)
    'pol_l': torch.tensor([-0.2, +0.3, +0.4]),   # log π_θ on dispreferred
    'ref_w': torch.tensor([ 0.0,  0.0,  0.0]),
    'ref_l': torch.tensor([ 0.0,  0.0,  0.0]),
}
margins = (batch['pol_w'] - batch['ref_w']) - (batch['pol_l'] - batch['ref_l'])
print('per-example margins:', margins.tolist())
print()
print(f'{"beta":>8}  {"DPO loss":>10}  {"accuracy":>10}')
for beta in [0.01, 0.1, 0.5, 2.0]:
    loss = dpo_loss(batch['pol_w'], batch['pol_l'], batch['ref_w'], batch['ref_l'], beta=beta)
    acc  = (margins > 0).float().mean().item()
    print(f'{beta:>8.2f}  {loss.item():>10.4f}  {acc:>10.2f}')

# Note: 'accuracy' here is a constant — it just checks the *sign* of the margin.
# β only changes the *loss magnitude*, not whether the preference is satisfied.

## Siguiente: `06_grpo.ipynb`

GRPO parte la diferencia — conserva el muestreo on-policy de PPO, pero descarta la función de valor.